In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')


In [ ]:
import asyncio, concurrent.futures, json, shutil, time, zipfile
import litellm, pandas as pd, torch, yaml
from pathlib import Path
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import EvalConfig, build_submission_limits, ALLOWED_ADAPTER_EXTENSIONS
from swegemma.evaluate import Evaluator
from swegemma.models import load_tasks

litellm.drop_params = True
DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
DECLARED_MODEL = 'gemma-4-e4b-it-qat-w4a16-ct'
MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{DECLARED_MODEL}/2')
ADAPTER_CANDIDATES = [
    Path('/kaggle/input/datasets/YOUR_KAGGLE_USERNAME/gemma4-e4b-lora-v1'),
    Path('/kaggle/input/gemma4-e4b-lora-v1'),
]
ADAPTER_SRC = next((p for p in ADAPTER_CANDIDATES if p.is_dir()), None)
assert ADAPTER_SRC is not None, f'Adapter dataset not found in {ADAPTER_CANDIDATES}'

SOUND24_IDS = ["fastapi_13537", "fastapi_13786", "fastapi_14349", "fastapi_14360", "fastapi_14487", "fastapi_14512", "fastapi_14873", "fastapi_14964", "fastapi_14978", "fastapi_15588", "fastapi_15763", "httpx_3672", "requests_6644", "requests_7315", "requests_7427", "requests_7505", "rich_3063", "rich_3468", "rich_3469", "rich_3506", "rich_3518", "rich_3676", "rich_3782", "rich_4077"]
LORA_VARIANTS = {
  "v3d_lora_seq3": {
    "agent.yaml": "agent_class: SequentialAgent\nname: gemma4_e4b_seq3_agent\ndescription: Three-stage fresh-context SWE pipeline (Locator -> Coder without submit_patch -> Adversarial Verifier/Fixer without write_file).\nsub_agents:\n  - config_path: sub_agents/locator.yaml\n  - config_path: sub_agents/coder.yaml\n  - config_path: sub_agents/verifier.yaml\n",
    "configs/sampling.yaml": "temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 4096\nthinking_config:\n  thinking_budget: 0\n  include_thoughts: false\n",
    "eval_config.yaml": "evaluation:\n  timeout_seconds: 240\n  max_tool_calls: 50\n  max_time_minutes: 8\n  max_turns: 100\n",
    "prompts/coder.md": "You are Stage 2 (Coder, fresh context) in `/workspace`.\nYour ONLY tools are: `run_command`, `read_file`, `edit_file`, `write_file`, `get_status`. Always invoke shell commands via `run_command(command=...)`.\nGoal: implement the minimal, exact source fix cleanly and compile-check it, then hand off to Stage 3.\n\n## Problem Statement\n{problem_description}\n\n## Stage 1 Location & Fix Plan\n{locate_plan?}\n\n## Steps\n1. Call `read_file` on the exact target file and line range from Stage 1 (if Stage 1 missed it, run ONE `git grep` via `run_command`).\n2. Call `edit_file` (short unique `old_string`, `allow_multiple=false`) on the source file(s). If `edit_file` fails twice, apply a short inline Python exact-replace via `run_command`.\n   - NEVER call `write_file` on `/tmp/...` or to create scratch/test scripts.\n3. Call `run_command` with `python3 -m py_compile <edited_files>` and optionally one quick inline `python3 -c \"...\"` smoke test.\n4. Stop once compiled and write a 4-line handoff: files edited, change applied, and edge cases Stage 3 should stress-test.\n\n## Hard Rules\n- NEVER modify test files (`tests/`, `test_*.py`, `*_test.py`) or configs (`conftest.py`, `pytest.ini`, `pyproject.toml`, `setup.cfg`, `tox.ini`).\n- NEVER run bare `pytest` or `pip install`.\n",
    "prompts/locator.md": "You are Stage 1 (Read-Only Locator) in `/workspace`.\nYour ONLY tools are `run_command` and `read_file`. Always run shell searches via `run_command(command=...)`.\nDO NOT edit any files. Use at most 6 tool calls, then immediately output a concise diagnosis report.\n\n## Steps\n1. Call `run_command` with a targeted search like `git grep -n -F -e '<Symbol1>' -e '<Symbol2>' -- '*.py' | head -25`.\n2. Call `read_file` on the most relevant 80\u2013120 line slice(s) of the source implementation file(s). For FastAPI doc-code issues, check `docs_src/`.\n3. Stop by call 6 and output a structured 10-line report:\n   - **Target file(s) and exact line numbers**\n   - **Exact symbol/function/class to change**\n   - **Root cause (2 sentences)**\n   - **Concrete minimal fix spec (exact behavior, return type, exception/message, edge cases)**\n",
    "prompts/verifier.md": "You are Stage 2 (Adversarial Verifier & Fixer) with a fresh context in `/workspace`.\nYour ONLY available tools are: `run_command`, `read_file`, `edit_file`, `get_status`, `submit_patch`. Always invoke shell commands through `run_command(command=...)`.\nStage 1 often edits the right file with a subtle bug, missed edge case, wrong exception/message, or incomplete logic. Audit critically, fix any flaw, clean stray files, and call `submit_patch`.\n\n## Problem Statement\n{problem_description}\n\n## Stage 1 Handoff\n{coder_notes?}\n\n## Steps\n1. **Audit current diff (call 1)**:\n   - Call `run_command` with `git status -s && git diff`.\n   - If the diff is empty, locate the source file immediately (`run_command` with `git grep`) and implement the fix with `edit_file`.\n   - If any untracked scratch files (`tmp/`, `verify_fix.py`, `repro.py`, `test_*.py`) or edits to protected test/config files exist in `/workspace`, clean/revert them via `run_command` (`rm -rf` / `git checkout --`).\n2. **Inspect full context (calls 2\u20134)**:\n   - Call `read_file` on 80\u2013120 lines around every edited function.\n   - Verify EVERY requirement in the problem statement: exact return value/type, exception type & message, default args, None/empty/boundary edge cases, and sibling callers.\n3. **Run focused verification (calls 5\u20138)**:\n   - Call `run_command` to run `python3 -m py_compile <edited_files>` AND a short inline `python3 -c \"...\"` script testing the reported bug + 2 edge cases + 1 normal control case.\n   - You may also run ONE targeted pytest node (`pytest tests/path/test_file.py -k <name>`) via `run_command`. If `pytest` is disabled, do NOT retry it\u2014use inline `python3 -c` assertions.\n   - NEVER write test files inside `/workspace` and NEVER loop more than twice debugging your own check script.\n4. **Repair if needed & submit**:\n   - If your audit or check reveals ANY defect, fix the source file with `edit_file` and re-run `py_compile` + inline check.\n   - Verify `git status -s` shows ONLY intended source edits (no stray files, no test/config changes).\n   - Call `submit_patch` as your final tool action while budget remains, confirm `patch_size > 0` and `files_changed > 0`, then give a 1-sentence summary.\n\n## Hard Rules\n- NEVER modify test files (`tests/`, `test_*.py`, `*_test.py`) or configs (`conftest.py`, `pytest.ini`, `pyproject.toml`, `setup.cfg`, `tox.ini`).\n- NEVER run bare `pytest`, `pip install`, or `git commit`.\n- NEVER finish without calling `submit_patch` on a compiled, non-empty source fix.\n",
    "sub_agents/coder.yaml": "name: coder_agent\nmodel: gemma-4-e4b-it-qat-w4a16-ct\ndescription: Stage 2 fresh-context coder that reads the located span and applies a clean, compiled source fix without submitting.\ninstruction: !include ../prompts/coder.md\ninclude_contents: none\noutput_key: coder_notes\ndisallow_transfer_to_parent: true\ndisallow_transfer_to_peers: true\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\ngenerate_content_config: !include ../configs/sampling.yaml\n",
    "sub_agents/locator.yaml": "name: locator_agent\nmodel: gemma-4-e4b-it-qat-w4a16-ct\ndescription: Stage 1 read-only locator that pinpoints exact source files, symbols, line ranges, and root cause.\ninstruction: !include ../prompts/locator.md\noutput_key: locate_plan\ndisallow_transfer_to_parent: true\ndisallow_transfer_to_peers: true\ntools:\n  - run_command\n  - read_file\ngenerate_content_config: !include ../configs/sampling.yaml\n",
    "sub_agents/verifier.yaml": "name: verifier_fixer_agent\nmodel: gemma-4-e4b-it-qat-w4a16-ct\ndescription: Stage 3 fresh-context adversarial verifier and fixer that audits git diff, runs focused checks, fixes defects, and calls submit_patch.\ninstruction: !include ../prompts/verifier.md\ninclude_contents: none\ndisallow_transfer_to_parent: true\ndisallow_transfer_to_peers: true\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - get_status\n  - submit_patch\ngenerate_content_config: !include ../configs/sampling.yaml\n"
  }
}
sub_dirs = {}
for name, files_map in LORA_VARIANTS.items():
    sdir = WORKING_DIR / f'sub_{name}'
    if sdir.exists():
        shutil.rmtree(sdir)
    for rel, txt in files_map.items():
        p = sdir / rel
        p.parent.mkdir(parents=True, exist_ok=True)
        if rel.endswith('.yaml') and 'model: gemma-4-e4b-it-qat-w4a16-ct' in txt:
            txt = txt.replace(
                'model: gemma-4-e4b-it-qat-w4a16-ct\n',
                'model: gemma-4-e4b-it-qat-w4a16-ct\nadapter: e4b_swe_lora\n',
            )
        p.write_text(txt, encoding='utf-8')
    adir = sdir / 'adapters' / 'e4b_swe_lora'
    adir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ADAPTER_SRC / 'adapter_config.json', adir / 'adapter_config.json')
    shutil.copy2(ADAPTER_SRC / 'adapter_model.safetensors', adir / 'adapter_model.safetensors')
    zout = WORKING_DIR / f'{name}_submission.zip'
    with zipfile.ZipFile(zout, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for fp in sorted(sdir.rglob('*')):
            if fp.is_file():
                zf.write(fp, arcname=str(fp.relative_to(sdir)))
    print(f'Packed {zout} ({zout.stat().st_size / 1e6:.1f} MB)', flush=True)
    sub_dirs[name] = sdir

first_dir = next(iter(sub_dirs.values()))
adapters = discover_adapters(str(first_dir), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=True,
    max_loras=1,
    max_lora_rank=16,
    scheduler_reserve_full_isl=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
    extra_args=(),
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server with LoRA started on {server_instance.base_url} (tp={tp_size})', flush=True)

models = server_instance.create_model_registry(
    aliases=[DECLARED_MODEL],
    model_prefix='openai/',
    api_key=os.environ.get('LOCAL_API_KEY', 'EMPTY'),
)

def run_sync(coro_or_fn, *args, **kwargs):
    fn = (lambda: coro_or_fn(*args, **kwargs)) if (callable(coro_or_fn) and not asyncio.iscoroutine(coro_or_fn)) else (lambda: coro_or_fn)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())

limits, gen_constraints = build_submission_limits()
tasks = load_tasks(DATA_DIR / 'tasks.jsonl')
subset = sorted([t for t in tasks if t.instance_id in set(SOUND24_IDS)], key=lambda t: t.instance_id)
assert len(subset) == 24

all_summaries = []
for run_name, sdir in sub_dirs.items():
    print(f'\n================ EVALUATING {run_name} ({len(subset)} sound tasks) ================', flush=True)
    ev_raw = yaml.safe_load((sdir / 'eval_config.yaml').read_text()) or {}
    ev = ev_raw.get('evaluation', ev_raw)
    rdir = WORKING_DIR / f'eval_{run_name}'
    eval_config = EvalConfig(
        tasks_path=DATA_DIR / 'tasks.jsonl',
        snapshots_dir=DATA_DIR / 'snapshots',
        results_dir=rdir,
        submission_dir=sdir,
        models=models,
        sandbox='subprocess',
        timeout_seconds=int(ev.get('timeout_seconds', 240)),
        verification_timeout_seconds=300,
        max_time_minutes=float(ev.get('max_time_minutes', 8.0)),
        max_tool_calls=int(ev.get('max_tool_calls', 50)),
        max_turns=int(ev.get('max_turns', 100)),
        limits=limits,
        generation_constraints=gen_constraints,
        adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2, token_threshold=14336, event_retention_size=5),
        graph_dir=str(DATA_DIR / 'graphs'),
        embeddings_dir=str(DATA_DIR / 'embeddings'),
        wheels_dir=DATA_DIR / 'wheels',
        verbose=False,
    )
    evaluator = Evaluator(eval_config)
    rows = []
    for idx, task in enumerate(subset, 1):
        t_s = time.perf_counter()
        try:
            res = run_sync(evaluator.evaluate_task, task=task, task_index=idx, total_tasks=len(subset))
            patch = (res.agent_patch or '').strip()
            row = dict(run=run_name, id=task.instance_id, repo=task.repo, resolved=bool(res.resolved),
                       patch_chars=len(patch), tool_calls=res.tool_calls,
                       agent_seconds=round(float(res.duration_seconds or 0.0), 1),
                       test_exit_code=res.test_exit_code, error=getattr(res, 'error', None))
        except Exception as exc:
            row = dict(run=run_name, id=task.instance_id, repo=task.repo, resolved=False, patch_chars=0,
                       tool_calls=None, agent_seconds=None, test_exit_code=None, error=repr(exc)[:300])
        row['wall_seconds'] = round(time.perf_counter() - t_s, 1)
        rows.append(row)
        print(f"[{run_name} {idx}/{len(subset)}] {row['id']}: resolved={row['resolved']} patch={row['patch_chars']} calls={row['tool_calls']} agent={row['agent_seconds']}s err={row['error']}", flush=True)
    df = pd.DataFrame(rows)
    summary = dict(
        run=run_name, tasks=len(df), resolved=int(df['resolved'].sum()),
        resolution_rate=round(float(df['resolved'].mean()), 3),
        no_patch=int((df['patch_chars'] == 0).sum()),
        errors=int(df['error'].notna().sum()),
        mean_tool_calls=round(float(df['tool_calls'].dropna().mean()), 1) if len(df['tool_calls'].dropna()) else None,
        agent_seconds_median=float(df['agent_seconds'].dropna().median()) if len(df['agent_seconds'].dropna()) else None,
        wall_seconds_mean=round(float(df['wall_seconds'].mean()), 1),
    )
    (WORKING_DIR / f'{run_name}_per_task.jsonl').write_text('\n'.join(json.dumps(r) for r in rows) + '\n')
    all_summaries.append(summary)
    print(json.dumps(summary, indent=2), flush=True)
    print(df.to_string(index=False), flush=True)

(WORKING_DIR / 'lora_eval24_summary.json').write_text(json.dumps(all_summaries, indent=2))
print('\n=== FINAL 24-TASK LORA COMPARISON ===', flush=True)
print(pd.DataFrame(all_summaries).to_string(index=False), flush=True)
